# Perplexity Experimentation for Different Models

I want to explore the differences in perplexity across different models across different datasets. This will setup some of my work with perplexity for token prediction, etc. I will be testing on these models
- GPT2
- Llama3-8B
- Llama2-7B
- Mistral-7B
- Gemma-7B
- OpenChat3.6-8B

In [37]:
import torch as t
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
import altair as alt
from huggingface_hub import login
from tqdm import tqdm

import os
from dotenv import load_dotenv

device = "cuda" if t.cuda.is_available() else "cpu"

In [38]:
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
login(HF_TOKEN)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /Users/kj3moraes/.cache/huggingface/token
Login successful


In [39]:
GPT2_MODEL = "openai-community/gpt2-medium"
MISTRAL_MODEL = "mistralai/Mistral-7B-v0.3"
LLAMA3_MODEL = "meta-llama/Meta-Llama-3-8B"
LLAMA2_MODEL = "meta-llama/Meta-Llama-2-7b-hf"
GEMMA_MODEL = "google/gemma-7b"
OPENCHAT_MODEL = "openchat/openchat-3.6-8b-20240522"

## Datasets 

I will be using 3 different datasets to test out 
- Salesforce/wikitext
- openai/openai_humaneval
- meta-math/MetaMathQA

We will evaluate all the models on a dataset by dataset basis. 

### Wikitext Evals

We will run all the models on wikitext and figure out the perplexities. We first get the **entire dataset** and compare the model's outputs to the actual text.

In [33]:
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
data = "\n".join(dataset["text"][:100])
print(f"The number of words in the wikitext dataset are {len(data.split(' '))}")

wikitext_ppls = dict() 

The number of words in the the wikitext dataset are 4781


In [31]:
def calculate_perplexity(model, tokenized_dataset, context_len, sequence_len, stride=512):
    nlls = []
    prev_end_loc = 0
    for start_loc in tqdm(range(0, sequence_len, stride)):
        end_loc = start_loc + context_len
        tgt_loc = end_loc - prev_end_loc

        # tokenize the inputs 
        input_ids = tokenized_dataset.input_ids[:, start_loc:end_loc] 
        target_ids = input_ids.clone() 
        target_ids[:,:-tgt_loc] = -100 

        # calculate the model's loss
        with t.no_grad():
            outputs = model(input_ids, labels=target_ids)
            nll = outputs.loss
            nlls.append(nll)

        prev_end_loc = end_loc 
        if end_loc == sequence_len:
            break
    return t.exp(t.tensor(nlls).mean())

**GPT2 Evals**

In [16]:
model = AutoModelForCausalLM.from_pretrained(GPT2_MODEL)
tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL)

In [35]:
context_len = model.config.n_positions
tok_dataset = tokenizer(data, return_tensors='pt')
sequence_len = tok_dataset.input_ids.shape[1]
ppl = calculate_perplexity(model, tok_dataset, context_len=context_len, sequence_len=sequence_len, stride=512)
print(f"The perplexity of {GPT2_MODEL} is {ppl:.2f}")
wikitext_ppls[GPT2_MODEL] = ppl 

100%|██████████| 11/11 [00:17<00:00,  1.55s/it]

The perplexity of openai-community/gpt2-medium is 22.920875549316406


**Mistral Evals**

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MISTRAL_MODEL, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MISTRAL_MODEL)

In [ ]:
context_len = model.config.max_position_embeddings
tok_dataset = tokenizer(data, return_tensors='pt')
sequence_len = tok_dataset.input_ids.shape[1]
ppl = calculate_perplexity(model, tok_dataset, context_len=context_len, sequence_len=sequence_len, stride=512)
print(f"The perplexity of {MISTRAL_MODEL} is {ppl:.2f}")
wikitext_ppls[MISTRAL_MODEL] = ppl

**Llama3 Evals**

In [ ]:
model = AutoModelForCausalLM.from_pretrained(LLAMA3_MODEL, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(LLAMA3_MODEL)

In [ ]:
context_len = model.config.max_position_embeddings
tok_dataset = tokenizer(data, return_tensors='pt')
sequence_len = tok_dataset.input_ids.shape[1]
ppl = calculate_perplexity(model, tok_dataset, context_len=context_len, sequence_len=sequence_len, stride=512)
print(f"The perplexity of {LLAMA3_MODEL} is {ppl:.2f}")
wikitext_ppls[LLAMA3_MODEL] = ppl

**Llama2 Evals**

In [ ]:
model = AutoModelForCausalLM.from_pretrained(LLAMA2_MODEL, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(LLAMA2_MODEL)

In [ ]:
context_len = model.config.max_position_embeddings
tok_dataset = tokenizer(data, return_tensors='pt')
sequence_len = tok_dataset.input_ids.shape[1]
ppl = calculate_perplexity(model, tok_dataset, context_len=context_len, sequence_len=sequence_len, stride=512)
print(f"The perplexity of {LLAMA2_MODEL} is {ppl:.2f}"
wikitext_ppls[LLAMA2_MODEL] = ppl

**Gemma 7B Evals**

In [ ]:
model = AutoModelForCausalLM.from_pretrained(GEMMA_MODEL, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(GEMMA_MODEL)

In [ ]:
context_len = model.config.max_position_embeddings
tok_dataset = tokenizer(data, return_tensors='pt')
sequence_len = tok_dataset.input_ids.shape[1]
ppl = calculate_perplexity(model, tok_dataset, context_len=context_len, sequence_len=sequence_len, stride=512)
print(f"The perplexity of {GEMMA_MODEL} is {ppl:.2f}")
wikitext_ppls[GEMMA_MODEL] = ppl

**OpenChat-3.6 8B Evals**

In [43]:

tokenizer = AutoTokenizer.from_pretrained(OPENCHAT_MODEL)
tokenizer.eos_token_id

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


128009

In [ ]:
context_len = model.config.max_position_embeddings
tok_dataset = tokenizer(data, return_tensors='pt')
sequence_len = tok_dataset.input_ids.shape[1]
ppl = calculate_perplexity(model, tok_dataset, context_len=context_len, sequence_len=sequence_len, stride=512)
print(f"The perplexity of {OPENCHAT_MODEL} is {ppl:.2f}")
wikitext_ppls[OPENCHAT_MODEL] = ppl

## HumanEvals

We repeat the same thing here, but now instead of comparing the nll of the  